<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Autonomy Limits and Guardrails </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to build **safe and controlled agents** by implementing **autonomy limits and guardrails**.
This notebook shows practical techniques to prevent agents from performing dangerous actions, running indefinitely, or going off-track.

**Key Topics Covered Today:**
- Tool allowlists (whitelisting only safe tools)
- Hard limits on iterations and execution time
- Human-in-the-loop confirmation for risky actions
- Safe fallback responses when guardrails are triggered
- Balancing agent autonomy with responsibility

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" beautifulsoup4 requests duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/13

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.7 MB/s eta 0:00:00


## 2. Understanding Agent Guardrails

Even powerful agents can:
- Call dangerous tools (email, delete files, financial transactions)
- Run in infinite loops
- Perform harmful or unintended actions
- Expose sensitive information

**Guardrails** help us constrain autonomy while keeping the agent useful.

Today we implement four layers of safety:
1. **Tool Allowlist** – Only approved tools are allowed
2. **Iteration & Time Limits** – Prevent runaway behavior
3. **Human Confirmation** – Approval required for risky actions
4. **Safe Fallbacks** – Graceful responses when limits are hit

## 3. Defining Safe Tools vs Risky Tools

In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# Safe tools (always allowed)
search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform basic math calculations. Safe tool."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

# Improved risky tool - body is now optional with a default
@tool
def send_email(to: str, subject: str, body: str = "Hello from WeCloudData AI Agents demo - Day 4") -> str:
    """Simulated email sending tool.
    Requires 'to' and 'subject'. 'body' is optional."""
    print(f"[RISKY] Would send email to: {to}")
    print(f"Subject: {subject}")
    print(f"Body: {body[:200]}...")   # truncate for safety
    return f"Email would have been sent to {to} with subject '{subject}'"

@tool
def delete_file(filename: str) -> str:
    """Simulated dangerous action"""
    print(f"[DANGEROUS] Would delete file: {filename}")
    return f"File {filename} would have been deleted"

# Tool allowlist
safe_tools = [search_tool, calculate_math]
all_tools = safe_tools + [send_email, delete_file]

print("Safe tools:", [t.name for t in safe_tools])
print("All tools (including risky):", [t.name for t in all_tools])

Safe tools: ['Web_Search', 'calculate_math']
All tools (including risky): ['Web_Search', 'calculate_math', 'send_email', 'delete_file']


## 4. Creating a Guarded Agent with Tool Allowlist

In [10]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Pull standard ReAct prompt
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

# Guarded agent - only safe tools
guarded_agent = create_react_agent(llm=llm, tools=safe_tools, prompt=prompt)

guarded_executor = AgentExecutor(
    agent=guarded_agent,
    tools=safe_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,           # Strict limit
    max_execution_time=60       # Safety timeout (seconds)
)

print("Guarded agent created with tool allowlist and strict limits")

Guarded agent created with tool allowlist and strict limits


## 5. Demo 1: Safe Behavior with Tool Allowlist

In [11]:
safe_query = """Find the current population of Canada and Mexico in 2026,
then calculate their combined population and the percentage Canada represents of the total."""

response = guarded_executor.invoke({"input": safe_query})

print("\n" + "="*60)
print("FINAL ANSWER (Safe Agent):")
print(response["output"])



> Entering new AgentExecutor chain...
To find the current population of Canada and Mexico in 2026, I will need to search for the most recent population estimates for both countries. After that, I can calculate their combined population and determine the percentage that Canada represents of the total. 

Action: Web_Search  
Action Input: "current population of Canada 2026"  Statistics Canada conducts a country-wide census that collects demographic data every five years on the first and sixth year of each decade. The 2021 Canadian census enumerated a total population of 36,991,981, an increase of around 5.2 percent over the 2016 figure. The current population of Canada is 40,133,190 as of Saturday, May 23, 2026. Population clock live, current, historical and projected population.The total area of Canada is 9,984,670 km2 (3,855,103 mi2) according to the United Nations Statistics Division . Canada is a country in North America, known for the Rocky Mountains and vast Arctic tundra. It has

## 6. Human-in-the-Loop Confirmation for Risky Actions

In [12]:
from langchain_core.tools import BaseTool
from typing import Dict, Any

def risky_tool_wrapper(tool: BaseTool):
    """Wrap risky tools to require human confirmation"""
    original_func = tool.func

    def wrapped_func(*args, **kwargs):
        print(f"\n  RISKY ACTION DETECTED: {tool.name}")
        print(f"Arguments: {kwargs}")
        confirmation = input("Do you want to allow this action? (yes/no): ").strip().lower()

        if confirmation in ["yes", "y"]:
            return original_func(*args, **kwargs)
        else:
            return f"Action {tool.name} was blocked by user guardrail."

    tool.func = wrapped_func
    return tool

# Apply confirmation wrapper
risky_send_email = risky_tool_wrapper(send_email)
risky_delete_file = risky_tool_wrapper(delete_file)

print("Risky tools wrapped with human confirmation guardrail")

Risky tools wrapped with human confirmation guardrail


## 7. Demo 2: Testing Risky Actions with Guardrails

In [13]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Better prompt for tool-calling agents
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that uses tools safely. "
               "Always use the exact tool arguments required. Never invent email addresses."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create agent with structured tool calling (much more reliable)
risky_agent = create_tool_calling_agent(
    llm=llm,
    tools=[search_tool, calculate_math, risky_send_email],
    prompt=prompt
)

risky_executor = AgentExecutor(
    agent=risky_agent,
    tools=[search_tool, calculate_math, risky_send_email],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

risky_query = """Find the current CEO of OpenAI in 2026.
Then send a short polite email to them with:
- Subject: Greeting from WeCloudData Demo
- Body: Hello, I hope this message finds you well. This is a test from the WeCloudData AI Agents course - Day 4."""

print("Running query that will trigger risky action (fixed version)...")
response_risky = risky_executor.invoke({"input": risky_query})

print("\n" + "="*70)
print("FINAL ANSWER (with Risky Guardrail):")
print(response_risky["output"])

Running query that will trigger risky action (fixed version)...


> Entering new AgentExecutor chain...

Invoking: `Web_Search` with `{'query': 'current CEO of OpenAI 2026'}`


OpenAI CEO Sam Altman said on Tuesday the rapid development and adoption of AI would not lead to a global “jobs apocalypse” and the technology had not claimed as many white-collar jobs as he ... Executive shifts at OpenAI are often fast-moving, and its org chart can be fluid. Last week, Instagram's head of partnerships, Charles Porch, announced he had joined OpenAI. Sam Altman is (1) the CEO of OpenAI, (2) the executive who steered the launch of ChatGPT in November 2022, (3) the former president of startup accelerator Y Combinator, (4) a billionaire venture investor worth an estimated $2 to $3.3 billion as of May 2026, and (5) one of the most influential people in artificial intelligence. Sam is 41 years old and 5 feet 6 inches tall. He was born to a ... OpenAI is a series G company based in San Francisco (Unite

## 8. Safe Fallback Mechanism

In [14]:
def safe_fallback_handler(error: Exception, input_data: Dict[str, Any]) -> str:
    """Graceful fallback when agent fails or hits limits"""
    print(f"\n  Guardrail triggered: {str(error)}")
    return (
        "I am unable to complete this request due to safety restrictions. "
        "Please rephrase your query to use only safe operations (web search and math calculations)."
    )

# Enhanced safe executor
safe_executor = AgentExecutor(
    agent=guarded_agent,
    tools=safe_tools,
    verbose=True,
    handle_parsing_errors=safe_fallback_handler,
    max_iterations=5,      # Even stricter
    max_execution_time=30
)

print("Safe fallback handler configured")

Safe fallback handler configured


## 9. Demo 3: Triggering Safe Fallback

In [15]:
# Query designed to potentially hit iteration limits
fallback_query = """Keep searching and doing calculations repeatedly about world population trends
until you find something very interesting, then continue the process many times."""

print("Running query that may hit iteration limit...")
response_fallback = safe_executor.invoke({"input": fallback_query})

print("\n" + "="*60)
print("FINAL ANSWER (with Fallback Guardrail):")
print(response_fallback["output"])

Running query that may hit iteration limit...


> Entering new AgentExecutor chain...
I need to start by searching for current world population trends to find something interesting. I'll look for recent statistics or projections that might reveal significant changes or insights. 

Action: Web_Search  
Action Input: "current world population trends 2023"  2 weeks ago - Based on this, the UN projected that the world population, 8 billion as of 2023, would peak around the year 2084 at about 10.3 billion, and then start a slow decline, assuming a continuing decrease in the global average fertility rate from 2.5 ... 3 weeks ago - The 2019 Revision of the UN estimates ... projected that the global population will reach 8 billion by 2023 as it increases by 156 every minute.... April 28, 2026 - World population has reached 8 ... International Database (IDB) estimates instead that the world population hit 8 billion on September 26, 2023...... 1 day ago - If fertility falls by more than projecte

## 10. Summary of Guardrails Implemented

 **Tool Allowlist** – Only safe tools are enabled by default  
 **Iteration & Time Limits** – `max_iterations` and `max_execution_time` prevent infinite loops  
 **Human-in-the-Loop** – Risky actions require explicit user approval  
 **Safe Fallbacks** – Agent responds gracefully when limits are reached  

---

## 11. Reflection Questions

1. Why is a tool **allowlist** more secure than a blocklist?
2. When would you require human confirmation vs. completely blocking a tool?
3. How would you extend these guardrails for a customer-facing production agent?
4. What other risky actions might need guardrails (code execution, database writes, payments, etc.)?
5. How do these safety patterns help balance agent autonomy with responsibility?